In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:

from wringlet import (
    data_provenance_enabled, 
    data_provenance_session_builder,
    provenance_column_name
)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    # SparkSession
    # .builder
    data_provenance_session_builder()
    .appName("data-provenance-notebook")
    .getOrCreate()
)

### Create toy dataframe

In [ ]:
from datetime import date
import pandas as pd

df = spark.createDataFrame([
    ("A", date(2026, 1, 15), 10.0, 90),
    ("A", date(2026, 1, 16), 10.0, 120),
    ("A", date(2026, 1, 17), 5.0, 300),
    ("B", date(2026, 1, 15), 100.0, 20),
    ("B", date(2026, 1, 16), 100.0, 30),
    ("B", date(2026, 1, 17), 80.0, 60),
], ["product", "date", "price", "sales"]
)
df.printSchema()

df.toPandas()

### Test with pyspark syntax

In [ ]:
df2 = df.select("product")

df2.toPandas()

In [ ]:
with data_provenance_enabled(spark, df) as df_with_provenance:
    res = df_with_provenance.toPandas()

res

### Test with SQL syntax

In [ ]:
df.createOrReplaceTempView("sales")

spark.sql("select * from sales").toPandas()

In [ ]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark, "sales") as view_with_provenance:
    res = spark.table(view_with_provenance).toPandas()

res

### Test with multiple dataframes/views and with custom name for provenance column

In [ ]:
spark.conf.set("spark.provenance.columnName", "toto")
print(provenance_column_name(spark))

with data_provenance_enabled(spark, df, "sales", df2) as (df_with_provenance, view_with_provenance, df2_with_provenance):
    res = df_with_provenance.toPandas()
    res2 = spark.table(view_with_provenance).toPandas()
    res3 = df2_with_provenance.toPandas()

print(res)
print(res2)
res3
